In [1]:
import os
import torch
import torch.nn as nn
import torchvision.models as models
import netvlad
from utils.pruning.common import mobilenetv2_L_block_pruning
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
encoder = models.mobilenet_v2(pretrained=False)
encoder  = encoder.features
layers = list(encoder.children())[:-1]


for block in layers[:-1]:
        for p in block.parameters():
            p.requires_grad = False

encoder = nn.Sequential(*layers)
model = nn.Module() 
model.add_module('encoder', encoder)
net_vlad = netvlad.NetVLAD(num_clusters=64, dim=320)
model.add_module('pool', net_vlad)


  
total_params = 0
zero_params = 0
trian_params =0

for p in model.parameters():
    num = p.numel()
    total_params += num

    if p.requires_grad:
        trian_params += p.numel()
   
   

print(f"Total parameters: {total_params}")
print(f"Train parameters: {trian_params}")



Total parameters: 1852672
Train parameters: 514880


In [3]:
pth_path = "runs/Aug20_07-58-18_mobilenetv2_netvlad"
resume_ckpt = os.path.join(pth_path, 'checkpoints', 'model_best.pth.tar')
checkpoint = torch.load(resume_ckpt, map_location=lambda storage, loc: storage)
model.load_state_dict(checkpoint['state_dict'])
model = model.to("cuda:0")

In [4]:
total_params = sum(p.numel() for p in model.parameters())
trian_params = sum(p.numel() for p in model.parameters() if p.requires_grad == True) 
zero_params = sum((p == 0).sum().item() for p in model.parameters())


for name, module in model.encoder[-1].named_modules():
    if isinstance(module, nn.Conv2d):
        print('='*10,name,"="*10)
        total_params = sum(p.numel() for p in module.parameters())
        zero_params = sum((p == 0).sum().item() for p in module.parameters())

        print(total_params)
        print(zero_params)





========== conv.0.0 ==========
153600
0
========== conv.1.0 ==========
8640
0
========== conv.2 ==========
307200
46080


In [5]:
from utils.pruning.common import mobilenetv2_L_block_reduction



reduction_model = mobilenetv2_L_block_reduction(model)


for name, module in reduction_model.encoder[-1].named_modules():
    if isinstance(module, nn.Conv2d):
        print(module.weight.shape)
       




torch.save(reduction_model, "reduced_model.pth")
print(reduction_model)
print(reduction_model.pool.centroids.shape)




torch.Size([960, 160, 1, 1])
torch.Size([960, 1, 3, 3])
torch.Size([272, 960, 1, 1])
Module(
  (encoder): Sequential(
    (0): ConvNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): ConvNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): ConvNormActivation(
          (0): Conv2d(16, 96, kernel_si

In [ ]:
#[72, 3, 480, 640] input shape
import torch.nn.functional as F

#dummy_input = torch.randn(72, 3, 480, 640).to("cuda:0")



#dummy_output = model.encoder(dummy_input)
#print(dummy_output.shape)# ([72, 320, 15, 20])
#dummy_output = model.pool(dummy_output)
# print(dummy_output.shape)

# print(dummy_output.shape)

dummy_input = torch.randn(72, 320, 15, 20).to("cuda:0")
N = 72
C = 320
x = F.normalize(dummy_input, p=2, dim=1)

print(x.shape) # ([72, 320, 15, 20])
soft_assign = model.pool.conv(x).view(N, 64, -1) # torch.Size([72, 64, 300])
soft_assign = F.softmax(soft_assign, dim=1) 

#print(soft_assign.shape)# torch.Size([72, 64, 300])
x_flatten = x.view(N, C, -1) # torch.Size([72, 320, 300])
#print(x_flatten.shape)
x_flatten =x_flatten.unsqueeze(0).permute(1, 0, 2, 3) # torch.Size([72, 320, 15, 20])
print(x_flatten.shape)
#print(model.pool.centroids.shape) # torch.Size([64, 320])




torch.Size([72, 320, 15, 20])
torch.Size([72, 1, 320, 300])


: 